# SafeVision · Notebook 1x · 8-class PPE detector (body part ↔ PPE)

**Goal:** extend the detector from 4 to **8 classes**, so SafeVision checks three PPE items per worker, each against the **body part it protects**:

| Body part | PPE | Why it matters |
|---|---|---|
| head | hard hat (helmet) | falling objects |
| hands | gloves | cuts, rebar, chemicals |
| face | safety glasses | grinding, welding, dust |
| person | high-visibility vest | vehicles and plant |

Most published PPE detectors only check hard hats and vests [2]. Checking **gloves and eye protection per worker**, by matching each item to the body part it should cover, is less common and is the extension studied here.

| ID | Question | Data | Main scores |
|---|---|---|---|
| **E1x** | 8-class YOLO11s at 960 px (best setting from E1b) | SH17 test split | mAP50, mAP50-95 per class |
| **E1x-TTA** | Does test-time augmentation help the small classes (gloves, glasses)? | SH17 test split | same |
| **E2** | Cross-dataset: person, helmet, vest | CHV (never trained on) | mAP50 |
| **E2b** | Cross-dataset: gloves, glasses (goggles), helmet, vest, person | Ultralytics Construction-PPE (never trained on) | mAP50 |

**Works on Kaggle and on Colab** (the notebook detects where it runs).

**Kaggle (recommended: free, runs in the background, SH17 needs no download):**
1. kaggle.com → **Create → New Notebook** → **File → Import Notebook** → upload this `.ipynb`.
2. Right panel → **Add Input** → search **SH17** → add the dataset.
3. Right panel → **Settings**: *Accelerator* = **GPU T4 x2** (or P100), *Internet* = **On** (needs a phone-verified Kaggle account).
4. **Add-ons → Secrets** → add `HF_TOKEN` (a Hugging Face *Write* token) and tick it for this notebook.
5. Top right **Save Version** → **Save & Run All (Commit)**. It now runs on Kaggle's servers; you can close the browser. When done, the ZIP is in the version's **Output** tab and also in your private HF repo.

**Colab:** `Runtime → Change runtime type → GPU`, set `KAGGLE_SLUG` in the Configuration cell, add the Secrets `KAGGLE_USERNAME`, `KAGGLE_KEY`, `HF_TOKEN`, then **Run all**.

**Rough time on a Kaggle T4:** data preparation 20-35 min · training at 960 px 2.5-3.5 h · evaluation 20 min.

**Outputs:** trained weights, a results Excel file (`E1x_E2_ppe8_detector.xlsx`), figures for the reports, all zipped at the end. We do **not** use Google Drive: everything is downloaded as a ZIP and can optionally be pushed to a private Hugging Face repository.

Paper numbers like **[2]** refer to the reference list at the end (same numbering as *SafeVision_Paper_Summaries.pdf*).

## 0. Check the GPU and install libraries

Shows which GPU Colab gave us and installs:
- **ultralytics**: the YOLO11 library we train with,
- **kaggle**: to download SH17,
- **gdown**: to download the public CHV dataset file,
- **openpyxl**: to write the results Excel file.

In [ ]:
# [colab-only]
# Show the GPU (expect A100 / L4 / T4). If this fails: Runtime -> Change runtime type -> GPU.
!nvidia-smi

# Install the libraries quietly. Ultralytics provides YOLO11 (detection and pose).
%pip install -q "ultralytics>=8.3.0" kaggle gdown openpyxl huggingface_hub

## 1. Configuration (edit this cell only)

All settings live here so every experiment is reproducible.

- **`KAGGLE_SLUG`**: the SH17 dataset on Kaggle. Open https://github.com/ahmadmughees/SH17dataset, click the Kaggle link in the README, and copy the part after `kaggle.com/datasets/` (format `owner/dataset-name`).
- **Classes:** we keep only the 4 SH17 classes we need [2]. Detecting a bare **head** lets us see "no helmet" directly instead of guessing it from a missing box.
- **Training settings** follow paper [12] (YOLO11n, 50 epochs, early stopping with patience 15) and the SH17 benchmark [2], which trained at 640 px.
- **Turn experiments on/off** with the `RUN_...` flags if time is short.

In [ ]:
# CONFIG
from pathlib import Path
import os, random

# ---- Where are we running? (Kaggle, Colab, or a local/lab PC such as the UOWD AI workstation) ----
import importlib.util
ON_KAGGLE = Path("/kaggle/input").exists()
IN_COLAB = importlib.util.find_spec("google.colab") is not None
ON_LOCAL = not (ON_KAGGLE or IN_COLAB)

# ---- Paths (local disk; fast, but wiped when the session ends) ----
if ON_KAGGLE:
    WORK = Path("/tmp/work")                               # big scratch space, not saved
    OUT  = Path("/kaggle/working/safevision_outputs")      # saved in the notebook's Output tab
elif IN_COLAB:
    WORK = Path("/content/work")                 # raw downloads + converted datasets
    OUT  = Path("/content/safevision_outputs")   # everything we keep (zipped at the end)
else:
    # Local / lab PC: by default under your user folder. Set SAFEVISION_DIR (e.g. D:\\safevision) to use another drive.
    BASE = Path(os.environ.get("SAFEVISION_DIR", Path.home() / "safevision"))
    WORK = BASE / "work"
    OUT  = BASE / "outputs"

# ---- Dataset sources ----
KAGGLE_SLUG = "PASTE_OWNER/PASTE_DATASET_NAME"  # SH17 on Kaggle, see the markdown above
CHV_GDRIVE_ID = "1fdGn67W0B7ShpBDbbQpUF0ScPQa4DR0a"  # public CHV file from the CHV GitHub README [3]

# ---- Our 8 target classes (order = class id 0..7). The first 4 keep the ids of the 4-class model. ----
CLASSES = ["person", "head", "helmet", "vest", "hands", "gloves", "face", "glasses"]

# SH17 class name -> our class name. Every other SH17 class (shoes, ear-muffs, ...) is dropped.
SH17_TO_OURS = {"person": "person", "head": "head", "helmet": "helmet", "safety-vest": "vest",
                "hands": "hands", "gloves": "gloves", "face": "face", "glasses": "glasses"}
CPPE_URL = "https://github.com/ultralytics/assets/releases/download/v0.0.0/construction-ppe.zip"   # E2b test set

# Fallback SH17 class order, used ONLY if no class-name file is found in the download.
# (Order as published with SH17 [2]; the visual check in Section 4 confirms it.)
SH17_DEFAULT_NAMES = ["person", "ear", "ear-mufs", "face", "face-guard", "face-mask-medical",
                      "foot", "tools", "glasses", "gloves", "helmet", "hands", "head",
                      "medical-suit", "shoes", "safety-suit", "safety-vest"]

# ---- Data preparation ----
SEED = 42              # same random split every time
MAX_SIDE = 1280        # SH17 photos are up to 8192 px wide [2]; shrinking them once saves a lot of training time
VAL_FRACTION = 0.10    # part of the SH17 training list kept aside to tune training (validation)

# ---- Training (paper [12]: 50 epochs, patience 15; paper [2]: 640 px) ----
EPOCHS = 50
PATIENCE = 15          # stop early if validation mAP does not improve for 15 epochs
BATCH = -1             # -1 = let Ultralytics pick the largest batch that fits the GPU
WORKERS = 8            # CPU threads loading images
DEVICE = 0             # first GPU

# ---- Safety net against Colab disconnects (needs the HF_TOKEN secret; see Section 3a) ----
CKPT_EVERY = 3         # upload the training checkpoint to your private HF repo every 3 epochs
CKPT_LOCAL_DIR = None  # (testing only) use a local folder instead of Hugging Face
FROM_CACHE = False     # set automatically to True when the converted dataset is found in your HF repo

# ---- Which experiments to run ----
RUN_E1B = False        # (4-class notebook only) not used here
RUN_E1C = False        # (4-class notebook only) not used here
RUN_E2  = True         # cross-dataset test on CHV (person, helmet, vest)
RUN_E2B = True         # cross-dataset test on Ultralytics Construction-PPE (gloves, goggles, ...)

# Folders used below
RAW = Path("/kaggle/input") if ON_KAGGLE else WORK / "raw_sh17"   # Kaggle: the attached SH17 input
DS  = WORK / "safevision_ppe8"         # converted 8-class SH17 dataset (YOLO format)
CHV_RAW = WORK / "raw_chv"
CHV_DS  = WORK / "chv_test"            # converted CHV test set (YOLO format)
RUNS = OUT / "runs"                    # Ultralytics training / validation runs
FIGS = OUT / "figures"
RES  = OUT / "results"
for p in [WORK, OUT, RUNS, FIGS, RES]:
    p.mkdir(parents=True, exist_ok=True)

random.seed(SEED)

def get_secret(name):
    # Read a secret from Colab Secrets, Kaggle Secrets or an environment variable (None if not set).
    # Keys are never printed.
    if os.environ.get(name):
        return os.environ[name]
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

print("Running on", "Kaggle" if ON_KAGGLE else "Colab" if IN_COLAB else "local/lab PC", "| work folder:", WORK, "| Classes:", CLASSES)

## 2. Results logger

Every number we report is written to **one table** (`experiments_log.csv`) with the experiment ID, settings and dataset. At the end the table is saved as the Excel file the team copies into the reports. This keeps every number traceable to the run that produced it.

In [ ]:
import csv, datetime, json
import pandas as pd

LOG_CSV = RES / "experiments_log.csv"
LOG_FIELDS = ["experiment", "run", "model", "imgsz", "dataset", "split",
              "class_name", "metric", "value", "notes", "timestamp"]

def log_result(experiment, run, model, imgsz, dataset, split, class_name, metric, value, notes=""):
    # Append one measured number to the shared results log (creates the file the first time).
    new_file = not LOG_CSV.exists()
    with open(LOG_CSV, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=LOG_FIELDS)
        if new_file:
            w.writeheader()
        w.writerow(dict(experiment=experiment, run=run, model=model, imgsz=imgsz,
                        dataset=dataset, split=split, class_name=class_name, metric=metric,
                        value=round(float(value), 4), notes=notes,
                        timestamp=datetime.datetime.now().isoformat(timespec="seconds")))

def read_log():
    # Load the whole results log as a table (empty table if nothing was logged yet).
    return pd.read_csv(LOG_CSV) if LOG_CSV.exists() else pd.DataFrame(columns=LOG_FIELDS)

print("Results will be logged to", LOG_CSV)

## 3a. Safety net: save progress to your private Hugging Face repo

Colab can disconnect at any time (idle timeout or usage limit) and then **everything in `/content` is deleted**. To avoid redoing hours of work:
- the converted 4-class dataset (about 2-3 GB) is uploaded once to a **private** dataset repo `safevision-sh17-4class`. In a new session, the notebook downloads that copy in a few minutes and **skips the 14 GB download and the slow conversion**.
- every `CKPT_EVERY` epochs the training checkpoint is uploaded to your private repo `safevision-artifacts`. After a disconnect, just **Run all** again: finished runs are skipped and an unfinished run **continues from its last checkpoint**.

Needs the secret `HF_TOKEN` (a *Write* token) in Colab Secrets or Kaggle Secrets. On Kaggle only the checkpoint part is used (SH17 is already attached). Without it the notebook still works, just without the safety net.

**Run order:** this section runs **before** the download. If it finds the cached dataset, Section 3 (download) and the conversion in Section 5 are skipped automatically.

In [ ]:
# [colab-only]
import zipfile
HF_API = None
try:
    from huggingface_hub import HfApi, hf_hub_download
    HF_API = HfApi(token=get_secret("HF_TOKEN"))
    HF_USER = HF_API.whoami()["name"]
    HF_REPO = f"{HF_USER}/safevision-artifacts"          # private model repo (checkpoints, outputs)
    DS_REPO = f"{HF_USER}/safevision-sh17-4class"        # private dataset repo (converted SH17)
    HF_API.create_repo(HF_REPO, private=True, exist_ok=True)
    HF_API.create_repo(DS_REPO, repo_type="dataset", private=True, exist_ok=True)
    print("Safety net ON. Checkpoints go to", HF_REPO)
except Exception as e:
    print("Safety net OFF (no HF_TOKEN secret?):", e)

if ON_KAGGLE:
    print("On Kaggle SH17 is attached as an input, so no dataset cache is needed.")
elif HF_API is not None and not (DS / "data.yaml").exists():
    try:
        z = hf_hub_download(DS_REPO, "safevision_ppe8.zip", repo_type="dataset", token=HF_API.token, local_dir=str(WORK / "dl"))
        print("Found the converted dataset in your HF repo. Unzipping...")
        zipfile.ZipFile(z).extractall(WORK)
        Path(z).unlink()                                   # free disk space
        FROM_CACHE = True
        print("Done: Section 3 (download) and the conversion in Section 5 will be skipped.")
    except Exception as e:
        print("No cached dataset yet (normal on the first run):", type(e).__name__)
elif (DS / "data.yaml").exists():
    FROM_CACHE = True
    print("Converted dataset already on this machine.")

## 3. Download SH17 from Kaggle

SH17 [2] has 8,099 worker photos and 75,994 labelled objects in 17 classes, already in YOLO format. Licence: non-commercial academic use (state this in the reports).

**Kaggle login:** the cell first looks for Colab *Secrets* named `KAGGLE_USERNAME` and `KAGGLE_KEY` (left sidebar → key icon). If they are not set, it asks you to upload your `kaggle.json` (Kaggle → Settings → *Create New Token*).

In [ ]:
# [colab-only]
import os, json, subprocess
from pathlib import Path

# Skip the download if the converted dataset came from the HF cache (Section 3a).
if ON_KAGGLE:
    print("On Kaggle: SH17 is read from the attached input under /kaggle/input (no download).")
elif FROM_CACHE:
    print("Converted dataset found: skipping the SH17 download.")
else:
    # 1) Kaggle credentials: secrets / environment variables, then ~/.kaggle/kaggle.json, then (Colab) upload.
    user, key = get_secret("KAGGLE_USERNAME"), get_secret("KAGGLE_KEY")
    if user and key:
        os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = user, key
        print("Using Kaggle credentials from secrets / environment variables.")
    elif (Path.home() / ".kaggle" / "kaggle.json").exists():
        print("Using", Path.home() / ".kaggle" / "kaggle.json")
    elif IN_COLAB:
        from google.colab import files
        print("Upload kaggle.json (Kaggle -> Settings -> Create New Token):")
        up = files.upload()
        creds = json.loads(list(up.values())[0])
        os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = creds["username"], creds["key"]
    else:
        raise RuntimeError(f"Put kaggle.json in {Path.home() / '.kaggle'} or set KAGGLE_USERNAME and KAGGLE_KEY.")

    # 2) Download and unzip SH17 into WORK/raw_sh17 (about 10-15 GB because of the large photos).
    assert "PASTE" not in KAGGLE_SLUG, "Set KAGGLE_SLUG in the Configuration cell first."
    RAW.mkdir(parents=True, exist_ok=True)
    subprocess.run(["kaggle", "datasets", "download", "-d", KAGGLE_SLUG, "-p", str(RAW), "--unzip"], check=True)
    print("Downloaded to", RAW)

## 4. Find the images, labels, class names and official split

Kaggle packages can be nested in different ways, so this cell **searches** for:
- the folder of YOLO label files (`.txt`) and the matching images,
- the class-name list (a `.yaml` with `names`, or `classes.txt`); if none is found we fall back to the published SH17 order and check it visually below,
- the official split lists `train_files.txt` / `val_files.txt` [2].

**Our split:** SH17's official *val* list becomes our **test** set (never used for training), and 10% of the official *train* list becomes our **validation** set (used during training). This keeps the test set honest.

In [ ]:
import yaml
from collections import Counter

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def find_label_dir(root):
    # Return the folder that holds the most YOLO .txt label files (ignoring split-list files).
    counts = Counter()
    for p in root.rglob("*.txt"):
        if p.name.lower() in {"train_files.txt", "val_files.txt", "test_files.txt", "classes.txt"}:
            continue
        counts[p.parent] += 1
    assert counts, f"No YOLO label files found under {root}"
    return counts.most_common(1)[0][0]

def index_images(root):
    # Map image file stem -> image path for every image under root.
    return {p.stem: p for p in root.rglob("*") if p.suffix.lower() in IMG_EXT}

def find_class_names(root, default):
    # Look for class names in a dataset .yaml or classes.txt; else use the published default order.
    for y in root.rglob("*.yaml"):
        try:
            data = yaml.safe_load(open(y))
            names = data.get("names") if isinstance(data, dict) else None
            if names:
                names = [names[k] for k in sorted(names)] if isinstance(names, dict) else list(names)
                print("Class names read from", y.name)
                return [str(n).strip().lower() for n in names]
        except Exception:
            pass
    for t in root.rglob("classes.txt"):
        names = [l.strip().lower() for l in open(t) if l.strip()]
        print("Class names read from", t.name)
        return names
    print("WARNING: no class-name file found; using the published SH17 order. Check the drawings in Section 6!")
    return [n.lower() for n in default]

def read_split_list(root, filename):
    # Read an official split list and return the set of image stems in it (empty set if missing).
    hits = list(root.rglob(filename))
    if not hits:
        return set()
    return {Path(l.strip()).stem for l in open(hits[0]) if l.strip()}

if FROM_CACHE:
    # Dataset restored from the HF cache: read the split from the folders.
    SPLITS = {sp: sorted(p.stem for p in (DS / "images" / sp).glob("*.jpg")) for sp in ["train", "val", "test"]}
    SH17_NAMES = [n.lower() for n in SH17_DEFAULT_NAMES]
    print("Using the cached converted dataset.")
else:
    LABEL_DIR = find_label_dir(RAW)
    IMAGES = index_images(RAW)
    SH17_NAMES = find_class_names(RAW, SH17_DEFAULT_NAMES)
    label_files = {p.stem: p for p in LABEL_DIR.glob("*.txt")}
    stems = sorted(set(label_files) & set(IMAGES))   # images that have a label file

    official_train = read_split_list(RAW, "train_files.txt")
    official_val   = read_split_list(RAW, "val_files.txt")

    rng = random.Random(SEED)
    if official_train and official_val:
        test_stems = sorted(s for s in stems if s in official_val)
        pool = sorted(s for s in stems if s in official_train)
        print("Using the official SH17 split (official val -> our test).")
    else:
        print("Official split lists not found: using a random 80/10/10 split.")
        pool = stems[:]
        rng.shuffle(pool)
        n_test = int(0.1 * len(pool))
        test_stems, pool = sorted(pool[:n_test]), pool[n_test:]
    rng.shuffle(pool)
    n_val = max(1, int(VAL_FRACTION * len(pool)))
    val_stems, train_stems = sorted(pool[:n_val]), sorted(pool[n_val:])

    SPLITS = {"train": train_stems, "val": val_stems, "test": test_stems}
    print(f"Label folder: {LABEL_DIR}")
    print(f"{len(SH17_NAMES)} source classes: {SH17_NAMES}")
print({k: len(v) for k, v in SPLITS.items()})

## 5. Convert SH17 to our 4-class dataset

For each image:
1. **Labels:** keep only person / head / helmet / safety-vest and renumber them 0-3 (`CLASSES`). YOLO labels are relative (0-1), so they stay valid after resizing.
2. **Image:** rotate according to the phone's EXIF orientation (so pixels match what the annotators saw), shrink so the longest side is at most `MAX_SIDE`, and save as JPEG.

Images with none of our 4 classes are kept as **background** examples; they teach the model not to fire on empty scenes. Runs in parallel threads.

In [ ]:
from PIL import Image, ImageOps
from concurrent.futures import ThreadPoolExecutor

src_to_ours = {}   # SH17 class id -> our class id
for i, n in enumerate(SH17_NAMES):
    if n in SH17_TO_OURS:
        src_to_ours[i] = CLASSES.index(SH17_TO_OURS[n])
missing = set(SH17_TO_OURS) - set(SH17_NAMES)
assert not missing, f"These SH17 classes were not found in the names list: {missing}"
print("SH17 id -> our id:", src_to_ours)

def convert_one(stem, split):
    # Write one resized image and its 4-class label file into DS/images/<split> and DS/labels/<split>.
    out_img = DS / "images" / split / f"{stem}.jpg"
    out_lbl = DS / "labels" / split / f"{stem}.txt"
    # Labels: keep and renumber our classes only.
    lines = []
    for line in open(label_files[stem]):
        parts = line.split()
        if len(parts) < 5:
            continue
        cls = int(float(parts[0]))
        if cls in src_to_ours:
            lines.append(" ".join([str(src_to_ours[cls])] + parts[1:5]))
    out_lbl.write_text("\n".join(lines) + ("\n" if lines else ""))
    # Image: fix orientation, shrink, save as JPEG.
    if not out_img.exists():
        im = ImageOps.exif_transpose(Image.open(IMAGES[stem])).convert("RGB")
        im.thumbnail((MAX_SIDE, MAX_SIDE))           # keeps the aspect ratio
        im.save(out_img, quality=92)
    return len(lines)

for split, ss in SPLITS.items():
    if FROM_CACHE:                                 # already converted (restored from the HF cache)
        print(f"{split}: {len(ss)} images (cached)")
        continue
    (DS / "images" / split).mkdir(parents=True, exist_ok=True)
    (DS / "labels" / split).mkdir(parents=True, exist_ok=True)
    with ThreadPoolExecutor(max_workers=16) as ex:
        n_boxes = sum(ex.map(lambda s: convert_one(s, split), ss))
    print(f"{split}: {len(ss)} images, {n_boxes} boxes")

# The dataset description file Ultralytics needs.
DATA_YAML = DS / "data.yaml"
yaml.safe_dump({"path": str(DS), "train": "images/train", "val": "images/val", "test": "images/test",
                "names": {i: n for i, n in enumerate(CLASSES)}}, open(DATA_YAML, "w"), sort_keys=False)
print("Wrote", DATA_YAML)

### 5a. Save the converted dataset to your private HF repo (once)
Runs in the **background** (training can start meanwhile). Next session, Section 3a downloads it instead of repeating Sections 3 and 5. The raw 14 GB download is deleted to free disk space.

In [ ]:
# [colab-only]
import threading, shutil, subprocess, zipfile
if HF_API is not None and not FROM_CACHE and not ON_KAGGLE:
    shutil.rmtree(RAW, ignore_errors=True)               # raw SH17 is no longer needed
    def _upload_dataset():
        try:
            zpath = WORK / "safevision_ppe8.zip"
            # -0 = store only (JPEGs are already compressed); fast to create and to unzip
            with zipfile.ZipFile(zpath, "w", zipfile.ZIP_STORED) as zf:      # works on Windows too (no 'zip' tool needed)
                for f in DS.rglob("*"):
                    if f.is_file():
                        zf.write(f, f.relative_to(WORK))
            HF_API.upload_file(path_or_fileobj=str(zpath), path_in_repo="safevision_ppe8.zip", repo_id=DS_REPO, repo_type="dataset")
            zpath.unlink()
            print("\n[background] Converted dataset saved to", DS_REPO)
        except Exception as e:
            print("\n[background] Dataset upload failed:", e)
    threading.Thread(target=_upload_dataset, daemon=True).start()
    print("Uploading the converted dataset in the background...")

## 6. Visual check of the labels (do not skip)

Draws the converted boxes on a few training images. **Check by eye** that green boxes are people, blue are heads, orange are helmets and purple are vests. If the colours are wrong, the class-name order was wrong: fix `SH17_DEFAULT_NAMES` and re-run Sections 4-5.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as patches

COLORS = {0: "lime", 1: "deepskyblue", 2: "orange", 3: "magenta", 4: "yellow", 5: "red", 6: "cyan", 7: "white"}
# person, head, helmet, vest, hands, gloves, face, glasses

def draw_yolo(ax, img_path, lbl_path, title=""):
    # Show an image with its YOLO boxes drawn in class colours.
    im = Image.open(img_path); W, H = im.size
    ax.imshow(im); ax.set_title(title, fontsize=8); ax.axis("off")
    if lbl_path.exists():
        for line in open(lbl_path):
            c, x, y, w, h = line.split()[:5]
            c, x, y, w, h = int(c), float(x) * W, float(y) * H, float(w) * W, float(h) * H
            ax.add_patch(patches.Rectangle((x - w / 2, y - h / 2), w, h, fill=False, lw=1.5, color=COLORS[c]))

sample = random.Random(SEED).sample(SPLITS["train"], k=min(6, len(SPLITS["train"])))
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, s in zip(axes.ravel(), sample):
    draw_yolo(ax, DS / "images/train" / f"{s}.jpg", DS / "labels/train" / f"{s}.txt", s)
fig.suptitle("Label check: green=person, blue=head, orange=helmet, purple=vest")
plt.tight_layout(); plt.savefig(FIGS / "fig_label_check.png", dpi=120); plt.show()

## 7. Dataset statistics (for the report)

Counts the boxes per class in each split. The table and chart go into the *Datasets* section of the vision report. SH17 is unbalanced and most objects are small [2], which is why we also test a larger input size (E1b).

In [ ]:
rows = []
for split in SPLITS:
    cnt = Counter()
    for p in (DS / "labels" / split).glob("*.txt"):
        for line in open(p):
            if line.strip():
                cnt[int(line.split()[0])] += 1
    rows.append({"split": split, "images": len(SPLITS[split]), **{CLASSES[i]: cnt.get(i, 0) for i in range(len(CLASSES))}})
stats = pd.DataFrame(rows).set_index("split")
stats.to_csv(RES / "dataset_stats_sh17_4class.csv")
display(stats) if "display" in globals() else print(stats)

ax = stats[CLASSES].T.plot(kind="bar", figsize=(8, 4), rot=0)
ax.set_ylabel("Number of boxes"); ax.set_title("SH17 (4 classes): boxes per class and split")
plt.tight_layout(); plt.savefig(FIGS / "fig_dataset_class_counts.png", dpi=150); plt.show()

## 8. Training and evaluation helpers

Two small functions so every experiment is run and measured **the same way**:

- `train_run(...)` trains one YOLO11 model and returns the best weights (chosen on the validation set).
- `evaluate(...)` measures a model on a split and logs overall and per-class **precision, recall, mAP50 and mAP50-95** (see *SafeVision_AI_Terms.pdf*). It also logs inference speed in ms per image.

Ultralytics reports precision and recall at the confidence threshold with the best F1, the standard YOLO setting.

In [ ]:
import torch
from ultralytics import YOLO

if not torch.cuda.is_available():
    DEVICE = "cpu"
    print("WARNING: no GPU found, training will be very slow.")

import shutil, threading

def remote_get(name, dest):
    # Download checkpoints/<name> from the private HF repo (or CKPT_LOCAL_DIR) to dest. True if found.
    try:
        if CKPT_LOCAL_DIR:
            src = Path(CKPT_LOCAL_DIR) / name
            if not src.exists():
                return False
            dest.parent.mkdir(parents=True, exist_ok=True); shutil.copy(src, dest); return True
        if globals().get("HF_API") is None:
            return False
        from huggingface_hub import hf_hub_download
        f = hf_hub_download(HF_REPO, f"checkpoints/{name}", token=HF_API.token)
        dest.parent.mkdir(parents=True, exist_ok=True); shutil.copy(f, dest); return True
    except Exception:
        return False

def remote_put(local, name, background=True):
    # Upload a file to checkpoints/<name> in the private HF repo (or CKPT_LOCAL_DIR).
    def _put():
        try:
            if CKPT_LOCAL_DIR:
                d = Path(CKPT_LOCAL_DIR) / name; d.parent.mkdir(parents=True, exist_ok=True); shutil.copy(local, d)
            elif globals().get("HF_API") is not None:
                HF_API.upload_file(path_or_fileobj=str(local), path_in_repo=f"checkpoints/{name}", repo_id=HF_REPO)
        except Exception as e:
            print(f"(checkpoint upload skipped: {e})")
        finally:
            if str(local).endswith(".upload"):
                Path(local).unlink(missing_ok=True)
    if background:
        tmp = Path(str(local) + ".upload"); shutil.copy(local, tmp); local = tmp   # copy so training can overwrite the original
        threading.Thread(target=_put, daemon=True).start()
    else:
        _put()

def train_run(run_name, base_model, imgsz, epochs=EPOCHS):
    # Train one model and return the path of its best checkpoint.
    # Safety net: skips runs that already finished and resumes unfinished runs from the last saved checkpoint.
    wdir = RUNS / run_name / "weights"
    if remote_get(f"{run_name}/best_final.pt", wdir / "best.pt"):
        remote_get(f"{run_name}/results.csv", RUNS / run_name / "results.csv")
        print(f"[{run_name}] already finished in an earlier session: using the saved best weights.")
        return wdir / "best.pt"

    def save_ckpt(trainer):
        # Called after every epoch's checkpoint is written; upload every CKPT_EVERY epochs.
        if (trainer.epoch + 1) % CKPT_EVERY == 0:
            remote_put(trainer.last, f"{run_name}/last.pt")
            if (Path(trainer.save_dir) / "results.csv").exists():   # keep the training curve too
                remote_put(Path(trainer.save_dir) / "results.csv", f"{run_name}/results.csv")

    if remote_get(f"{run_name}/last.pt", wdir / "last.pt"):
        remote_get(f"{run_name}/results.csv", RUNS / run_name / "results.csv")
        print(f"[{run_name}] resuming from the checkpoint saved before the disconnect.")
        model = YOLO(str(wdir / "last.pt"))
        model.add_callback("on_model_save", save_ckpt)
        model.train(resume=True)
    else:
        model = YOLO(base_model)                   # starts from COCO-pretrained weights (transfer learning)
        model.add_callback("on_model_save", save_ckpt)
        model.train(data=str(DATA_YAML), imgsz=imgsz, epochs=epochs, patience=PATIENCE,
                    batch=BATCH, workers=WORKERS, device=DEVICE, seed=SEED, deterministic=True,
                    project=str(RUNS), name=run_name, exist_ok=True, plots=True, verbose=False)
    best = Path(model.trainer.best)
    remote_put(best, f"{run_name}/best_final.pt", background=False)          # mark the run as finished
    if (RUNS / run_name / "results.csv").exists():
        remote_put(RUNS / run_name / "results.csv", f"{run_name}/results.csv", background=False)
    print(f"[{run_name}] best weights: {best}")
    return best

def evaluate(weights, data_yaml, split, imgsz, experiment, run_name, model_label, dataset_name,
             augment=False, keep_classes=None, notes=""):
    # Validate `weights` on `split` of `data_yaml`, log the scores, and return a summary dict.
    # keep_classes: list of class names to average over (e.g. CHV has no 'head' class).
    m = YOLO(str(weights))
    r = m.val(data=str(data_yaml), split=split, imgsz=imgsz, batch=16, device=DEVICE,
              augment=augment, plots=True, project=str(RUNS), name=f"val_{run_name}_{dataset_name}",
              exist_ok=True, verbose=False)
    names = r.names
    per_class = {}
    for i, c in enumerate(r.box.ap_class_index):     # only classes that appear in this split
        p, rec, ap50, ap = r.box.class_result(i)
        per_class[names[int(c)]] = dict(precision=p, recall=rec, mAP50=ap50, mAP50_95=ap)
    if keep_classes:                                   # average only over the classes this dataset has
        sel = [per_class[c] for c in keep_classes if c in per_class]
        overall = {k: float(sum(d[k] for d in sel) / len(sel)) for k in ["precision", "recall", "mAP50", "mAP50_95"]}
    else:
        overall = dict(precision=r.box.mp, recall=r.box.mr, mAP50=r.box.map50, mAP50_95=r.box.map)
    common = dict(experiment=experiment, run=run_name, model=model_label, imgsz=imgsz,
                  dataset=dataset_name, split=split)
    for k, v in overall.items():
        log_result(**common, class_name="all", metric=k, value=v, notes=notes)
    for c, d in per_class.items():
        for k, v in d.items():
            log_result(**common, class_name=c, metric=k, value=v, notes=notes)
    log_result(**common, class_name="all", metric="inference_ms_per_image", value=r.speed["inference"], notes=notes)
    print(f"[{experiment} | {run_name} | {dataset_name}/{split}] " +
          ", ".join(f"{k}={v:.3f}" for k, v in overall.items()))
    return {"overall": overall, "per_class": per_class}

summaries = {}   # run name -> evaluation summary (filled by the experiments below)

## 9. Experiment E1x · 8-class YOLO11s at 960 px

**Why 960 px:** in the 4-class experiments, 960 px beat 640 px (mAP50 0.746 vs 0.727), mostly on small objects. Gloves and glasses are even smaller, so we train directly at 960 px.

**Note for the report:** compare the per-class numbers for *gloves*, *glasses*, *hands* and *face* with SH17's per-class results [2]. Small objects are expected to score lower than heads and people. The first four classes can be compared with the 4-class model (E1b) to see whether adding classes changed them.

In [ ]:
# Name kept as best_e1a so the rest of the notebook works unchanged; it is the 8-class model at 960 px.
best_e1a = train_run("E1x_yolo11s_960_ppe8", "yolo11s.pt", imgsz=960)
summaries["E1x"] = evaluate(best_e1a, DATA_YAML, "test", 960, "E1x", "E1x_yolo11s_960_ppe8", "YOLO11s-8cls", "SH17")

## 10. (4-class notebook only) E1b

**Why:** about half of SH17 objects cover less than 1% of the image [2], and site cameras often see workers from far away. A larger input keeps more pixels per helmet. This is an **ablation**: same model, only the image size changes.

In [ ]:
if RUN_E1B:
    best_e1b = train_run("E1b_yolo11s_960", "yolo11s.pt", imgsz=960)
    summaries["E1b"] = evaluate(best_e1b, DATA_YAML, "test", 960, "E1", "E1b_yolo11s_960", "YOLO11s", "SH17")
else:
    print("E1b skipped (RUN_E1B = False)")

## 11. (4-class notebook only) E1c

**Why:** checks whether extra model capacity helps, at the cost of speed. Only run it if there is time (`RUN_E1C = True`).

In [ ]:
if RUN_E1C:
    best_e1c = train_run("E1c_yolo11m_960", "yolo11m.pt", imgsz=960)
    summaries["E1c"] = evaluate(best_e1c, DATA_YAML, "test", 960, "E1", "E1c_yolo11m_960", "YOLO11m", "SH17")
else:
    print("E1c skipped (RUN_E1C = False)")

## 12. Choose the best model, then Experiment E1d · test-time augmentation (TTA)

The **best model** is the one with the highest **validation** mAP50-95. We pick on validation, not test, so the test set stays a fair final exam.

**E1d** runs the best model again on the test set with **TTA** (flipped and rescaled copies of each photo, results merged). It needs no extra training and may add a little accuracy at the cost of speed.

In [ ]:
candidates = {"E1x": (best_e1a, 960)}
if RUN_E1B: candidates["E1b"] = (best_e1b, 960)
if RUN_E1C: candidates["E1c"] = (best_e1c, 960)

# Pick on the VALIDATION split (no logging, to keep the results table about test scores).
val_scores = {}
for k, (w, sz) in candidates.items():
    r = YOLO(str(w)).val(data=str(DATA_YAML), split="val", imgsz=sz, batch=16, device=DEVICE,
                         plots=False, verbose=False, project=str(RUNS), name=f"select_{k}", exist_ok=True)
    val_scores[k] = r.box.map
print("Validation mAP50-95:", {k: round(v, 4) for k, v in val_scores.items()})

BEST_KEY = max(val_scores, key=val_scores.get)
BEST_WEIGHTS, BEST_IMGSZ = candidates[BEST_KEY]
print(f"Best model: {BEST_KEY} ({BEST_WEIGHTS}, {BEST_IMGSZ}px)")

summaries["E1x_TTA"] = evaluate(BEST_WEIGHTS, DATA_YAML, "test", BEST_IMGSZ, "E1x", f"E1x_TTA_on_{BEST_KEY}",
                            f"{BEST_KEY}+TTA", "SH17", augment=True, notes="test-time augmentation")

## 13. Compare the E1 experiments (table + chart for the report)

One row per experiment with the overall test scores, plus the per-class mAP50, the numbers to compare with SH17's per-class results [2].

In [ ]:
log = read_log()
e1 = log[(log.experiment == "E1x") & (log.dataset == "SH17")]

overall = (e1[e1.class_name == "all"]
           .pivot_table(index=["run", "model", "imgsz"], columns="metric", values="value")
           .reset_index())
per_class = (e1[(e1.class_name != "all") & (e1.metric == "mAP50")]
             .pivot_table(index="run", columns="class_name", values="value"))
display(overall) if "display" in globals() else print(overall)
display(per_class) if "display" in globals() else print(per_class)

ax = per_class.plot(kind="bar", figsize=(9, 4), rot=0)
ax.set_ylabel("mAP50 (test)"); ax.set_ylim(0, 1); ax.set_title("E1x: per-class mAP50 on the SH17 test split (8 classes)")
plt.tight_layout(); plt.savefig(FIGS / "fig_E1_per_class_map50.png", dpi=150); plt.show()

## 14. Download the CHV dataset (for Experiment E2)

CHV [3] has 1,330 real construction-site photos with **person**, **vest** and helmets in **4 colours**. We **never train on it**: it shows how the detector works on photos from a different source (cross-dataset test). The file is public and linked from the CHV GitHub README; we only download it with `gdown`.

In [ ]:
# [colab-only]
import subprocess
if RUN_E2:
    CHV_RAW.mkdir(parents=True, exist_ok=True)
    zip_path = CHV_RAW / "chv.zip"
    subprocess.run(["gdown", "--fuzzy", f"https://drive.google.com/file/d/{CHV_GDRIVE_ID}/view", "-O", str(zip_path)], check=True)
    subprocess.run(["unzip", "-q", "-o", str(zip_path), "-d", str(CHV_RAW)], check=True)
    print("CHV extracted to", CHV_RAW)

## 15. Convert CHV to our classes

CHV labels may be **YOLO `.txt`** files or **Pascal VOC `.xml`** files; the cell handles both. Mapping: `person → person`, `vest → vest`, and every helmet colour (blue, red, white, yellow) → `helmet`. CHV has **no head class**, so E2 is scored on person, helmet and vest only.

All CHV images go into one `test` split. The visual check confirms the mapping.

In [ ]:
import xml.etree.ElementTree as ET

CHV_DEFAULT_NAMES = ["person", "vest", "blue", "red", "white", "yellow"]   # used only if no names file exists

def chv_name_to_ours(name):
    # Map a CHV class name to one of our class names (or None to drop it).
    n = name.lower()
    if "person" in n: return "person"
    if "vest" in n: return "vest"
    if "helmet" in n or n in {"blue", "red", "white", "yellow"}: return "helmet"
    return None

def convert_chv():
    # Build CHV_DS/images/test + CHV_DS/labels/test in our 4-class YOLO format.
    (CHV_DS / "images/test").mkdir(parents=True, exist_ok=True)
    (CHV_DS / "labels/test").mkdir(parents=True, exist_ok=True)
    # Skip macOS junk copies inside the ZIP (__MACOSX folders, '._name' files); they are not real images/labels.
    junk = lambda p: "__MACOSX" in p.parts or p.name.startswith("._")
    imgs = {k: v for k, v in index_images(CHV_RAW).items() if not junk(v)}
    xmls = {p.stem: p for p in CHV_RAW.rglob("*.xml") if not junk(p)}
    n_img = n_box = 0
    if xmls:                                                    # Pascal VOC format
        for stem, xp in xmls.items():
            if stem not in imgs: continue
            root = ET.parse(xp).getroot()
            W = float(root.find("size/width").text); H = float(root.find("size/height").text)
            lines = []
            for obj in root.findall("object"):
                ours = chv_name_to_ours(obj.find("name").text)
                if ours is None: continue
                b = obj.find("bndbox"); x1, y1, x2, y2 = (float(b.find(t).text) for t in ["xmin", "ymin", "xmax", "ymax"])
                lines.append(f"{CLASSES.index(ours)} {(x1+x2)/2/W:.6f} {(y1+y2)/2/H:.6f} {(x2-x1)/W:.6f} {(y2-y1)/H:.6f}")
            ImageOps.exif_transpose(Image.open(imgs[stem])).convert("RGB").save(CHV_DS / "images/test" / f"{stem}.jpg", quality=95)
            (CHV_DS / "labels/test" / f"{stem}.txt").write_text("\n".join(lines) + "\n")
            n_img += 1; n_box += len(lines)
    else:                                                       # YOLO txt format
        names = find_class_names(CHV_RAW, CHV_DEFAULT_NAMES)
        id_map = {i: CLASSES.index(chv_name_to_ours(n)) for i, n in enumerate(names) if chv_name_to_ours(n)}
        print("CHV id -> our id:", id_map)
        for lp in CHV_RAW.rglob("*.txt"):
            if junk(lp) or lp.stem not in imgs or lp.name == "classes.txt": continue
            lines = []
            for line in open(lp, encoding="utf-8", errors="ignore"):   # tolerate odd bytes in label files
                parts = line.split()
                try:
                    cid = int(float(parts[0])) if len(parts) >= 5 else None
                except ValueError:                                     # not a label line
                    continue
                if cid in id_map:
                    lines.append(" ".join([str(id_map[cid])] + parts[1:5]))
            ImageOps.exif_transpose(Image.open(imgs[lp.stem])).convert("RGB").save(CHV_DS / "images/test" / f"{lp.stem}.jpg", quality=95)
            (CHV_DS / "labels/test" / f"{lp.stem}.txt").write_text("\n".join(lines) + "\n")
            n_img += 1; n_box += len(lines)
    yaml.safe_dump({"path": str(CHV_DS), "train": "images/test", "val": "images/test", "test": "images/test",
                    "names": {i: n for i, n in enumerate(CLASSES)}}, open(CHV_DS / "data.yaml", "w"), sort_keys=False)
    print(f"CHV converted: {n_img} images, {n_box} boxes")

if RUN_E2:
    convert_chv()
    sample = random.Random(SEED).sample(sorted(p.stem for p in (CHV_DS / "images/test").glob("*.jpg")), k=3)
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, s in zip(axes, sample):
        draw_yolo(ax, CHV_DS / "images/test" / f"{s}.jpg", CHV_DS / "labels/test" / f"{s}.txt", s)
    fig.suptitle("CHV label check: green=person, orange=helmet, purple=vest")
    plt.tight_layout(); plt.savefig(FIGS / "fig_chv_label_check.png", dpi=120); plt.show()

## 16. Experiment E2 · Cross-dataset test on CHV

**Why:** a detector can look good on its own test split but fail on photos taken elsewhere (paper [2] reports helmet recall of only 23.7% on another dataset). E2 measures this honestly, with and without TTA.

**How to report it:** CHV's own paper reaches 86.55% mAP **when trained on CHV** [3]. Our model has **never seen CHV**, so a lower score is expected and is still a fair, useful result. Report it as cross-dataset generalisation. Note too that "person" boxes may be drawn slightly differently in SH17 and CHV.

In [ ]:
if RUN_E2:
    CHV_YAML = CHV_DS / "data.yaml"
    keep = ["person", "helmet", "vest"]                 # CHV has no 'head' labels
    summaries["E2"] = evaluate(BEST_WEIGHTS, CHV_YAML, "test", BEST_IMGSZ, "E2", f"E2_{BEST_KEY}_on_CHV",
                               BEST_KEY, "CHV", keep_classes=keep, notes="cross-dataset, never trained on CHV")
    summaries["E2_TTA"] = evaluate(BEST_WEIGHTS, CHV_YAML, "test", BEST_IMGSZ, "E2", f"E2_{BEST_KEY}_TTA_on_CHV",
                                   f"{BEST_KEY}+TTA", "CHV", augment=True, keep_classes=keep,
                                   notes="cross-dataset + test-time augmentation")
else:
    print("E2 skipped (RUN_E2 = False)")

## 16b. Experiment E2b · Cross-dataset test on Ultralytics Construction-PPE (gloves and goggles)

**Why:** CHV (E2) has no gloves or glasses. Ultralytics **Construction-PPE** (November 2026 release, 1,000 images, CC BY 4.0) is a recent construction dataset labelled with helmet, gloves, vest, boots, goggles, person and forklift (plus "missing PPE" classes). Our model has **never seen it**, so this measures how well the new classes transfer to real construction sites. The current release has only train (800) and val (200) splits, so we test on its **val** split (a test split is used if one exists). Their "goggles" are matched to our "glasses"; "boots", "no_*" and other classes are ignored (our model has no such classes).

In [ ]:
CPPE_RAW, CPPE_DS = WORK / "raw_cppe", WORK / "cppe_test"
CPPE_MAP = {"helmet": "helmet", "gloves": "gloves", "vest": "vest", "goggles": "glasses", "person": "person"}
CPPE_DEFAULT = ["helmet", "gloves", "vest", "boots", "goggles", "none", "Person",
                "no_helmet", "no_goggle", "no_gloves", "no_boots"]

def cppe_names():
    # Class names of Construction-PPE: from the dataset yaml in the ZIP or in the ultralytics package.
    import ultralytics
    for y in list(CPPE_RAW.rglob("*.yaml")) + [Path(ultralytics.__file__).parent / "cfg/datasets/construction-ppe.yaml"]:
        try:
            names = yaml.safe_load(open(y))["names"]
            return [names[i] for i in sorted(names)] if isinstance(names, dict) else list(names)
        except Exception:
            continue
    print("WARNING: Construction-PPE class names not found; using the published order.")
    return CPPE_DEFAULT

if RUN_E2B:
    CPPE_RAW.mkdir(parents=True, exist_ok=True)
    import urllib.request, zipfile
    try:
        urllib.request.urlretrieve(CPPE_URL, CPPE_RAW / "cppe.zip")      # about 0.2-0.5 GB
        zipfile.ZipFile(CPPE_RAW / "cppe.zip").extractall(CPPE_RAW)
    except Exception as e:
        RUN_E2B = False
        print("Could not download Construction-PPE (", e, "): E2b skipped. Check CPPE_URL on docs.ultralytics.com.")
if RUN_E2B:
    names = cppe_names()
    id_map = {i: CLASSES.index(CPPE_MAP[n.lower()]) for i, n in enumerate(names) if n.lower() in CPPE_MAP}
    print("Construction-PPE classes:", names, "| id -> ours:", id_map)
    # The current release has only train/val (no test split); our model never saw either, so val is a fair test.
    lbl_dirs = ([d for d in CPPE_RAW.rglob("labels/test") if d.is_dir()] or [d for d in CPPE_RAW.rglob("labels/val") if d.is_dir()]
                or [d for d in CPPE_RAW.rglob("val/labels") if d.is_dir()] or [d for d in CPPE_RAW.rglob("valid/labels") if d.is_dir()])
    if not lbl_dirs:
        RUN_E2B = False
        print("E2b skipped: no labels/val folder found in the download. Folders:", sorted({str(p.parent.relative_to(CPPE_RAW)) for p in CPPE_RAW.rglob("*.txt")})[:10])
if RUN_E2B:
    lbl_dir = lbl_dirs[0]; img_dir = Path(str(lbl_dir).replace("labels", "images"))
    (CPPE_DS / "images/test").mkdir(parents=True, exist_ok=True); (CPPE_DS / "labels/test").mkdir(parents=True, exist_ok=True)
    n_img = 0
    for lp in lbl_dir.glob("*.txt"):
        imgs = [p for p in img_dir.glob(lp.stem + ".*") if p.suffix.lower() in (".jpg", ".jpeg", ".png")]
        if not imgs:
            continue
        lines = []
        for line in open(lp, encoding="utf-8", errors="ignore"):
            parts = line.split()
            try:
                c = int(float(parts[0]))
            except (ValueError, IndexError):
                continue
            if c in id_map and len(parts) >= 5:
                lines.append(" ".join([str(id_map[c])] + parts[1:5]))
        Image.open(imgs[0]).convert("RGB").save(CPPE_DS / "images/test" / f"{lp.stem}.jpg", quality=95)
        (CPPE_DS / "labels/test" / f"{lp.stem}.txt").write_text("\n".join(lines) + "\n")
        n_img += 1
    yaml.safe_dump({"path": str(CPPE_DS), "train": "images/test", "val": "images/test", "test": "images/test",
                    "names": {i: n for i, n in enumerate(CLASSES)}}, open(CPPE_DS / "data.yaml", "w"), sort_keys=False)
    print("Construction-PPE test images converted:", n_img, "from", lbl_dir)
    keep = ["person", "helmet", "vest", "gloves", "glasses"]
    summaries["E2b"] = evaluate(BEST_WEIGHTS, CPPE_DS / "data.yaml", "test", BEST_IMGSZ, "E2b", f"E2b_{BEST_KEY}_on_CPPE",
                                BEST_KEY, "Construction-PPE", keep_classes=keep, notes="cross-dataset, never trained on it")
else:
    print("E2b skipped (RUN_E2B = False)")

## 17. Example detections (figures for the slides and reports)

Runs the best model on a few test photos from SH17 (and CHV) and saves the annotated images. Figures are named `fig_detections_*.png`, as agreed with the team.

In [ ]:
def save_detection_grid(img_dir, out_name, k=6, title=""):
    # Predict on k random images from img_dir and save them as one grid figure.
    paths = sorted(img_dir.glob("*.jpg"))
    paths = random.Random(SEED).sample(paths, k=min(k, len(paths)))
    model = YOLO(str(BEST_WEIGHTS))
    preds = model.predict([str(p) for p in paths], imgsz=BEST_IMGSZ, conf=0.25, device=DEVICE, verbose=False)
    cols = 3; rows = (len(preds) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))
    for ax, pr in zip(axes.ravel(), preds):
        ax.imshow(pr.plot()[:, :, ::-1]); ax.axis("off")      # .plot() returns BGR; flip to RGB
    for ax in axes.ravel()[len(preds):]:
        ax.axis("off")
    fig.suptitle(title); plt.tight_layout()
    plt.savefig(FIGS / out_name, dpi=120); plt.show()

save_detection_grid(DS / "images/test", "fig_detections_sh17.png", title=f"{BEST_KEY} on SH17 test photos")
if RUN_E2:
    save_detection_grid(CHV_DS / "images/test", "fig_detections_chv.png", title=f"{BEST_KEY} on CHV (never trained on)")

## 18. Save everything: Excel results, best model, ZIP download

- `results/E1x_E2_ppe8_detector.xlsx`: one sheet per experiment (E1, E2) plus the full log. Ralansha copies the numbers from here into the reports.
- `models/detector_best.pt`: the best detector, used by the backend and Notebook 2.
- `run_config.json`: the settings, for the "experimental setup" section of the reports.
- Everything is zipped and downloaded to your computer. Optionally, it is also pushed to a **private** Hugging Face repo if a `HF_TOKEN` secret exists (no Google Drive).

In [ ]:
import shutil

log = read_log()
xlsx = RES / "E1x_E2_ppe8_detector.xlsx"
with pd.ExcelWriter(xlsx) as xw:
    for exp in sorted(log.experiment.unique()):
        sub = log[log.experiment == exp]
        sub.pivot_table(index=["run", "model", "imgsz", "dataset", "split", "class_name"],
                        columns="metric", values="value").reset_index().to_excel(xw, sheet_name=exp, index=False)
    log.to_excel(xw, sheet_name="full_log", index=False)
print("Saved", xlsx)

(OUT / "models").mkdir(exist_ok=True)
shutil.copy(BEST_WEIGHTS, OUT / "models" / "detector_best.pt")
json.dump({"best_run": BEST_KEY, "best_imgsz": BEST_IMGSZ, "classes": CLASSES, "epochs": EPOCHS,
           "patience": PATIENCE, "seed": SEED, "max_side": MAX_SIDE, "val_fraction": VAL_FRACTION,
           "split_sizes": {k: len(v) for k, v in SPLITS.items()}},
          open(OUT / "run_config.json", "w"), indent=2)

archive = shutil.make_archive(str(OUT.parent / "safevision_nb1x_outputs"), "zip", OUT)
print("ZIP:", archive)

In [ ]:
# [colab-only]
# Download the ZIP to your computer, and optionally push the outputs to a PRIVATE Hugging Face repo.
if ON_KAGGLE:
    print("On Kaggle: download the ZIP from this version's Output tab.")
elif IN_COLAB:
    from google.colab import files
    files.download(archive)
else:
    print("Local/lab PC: the ZIP is at", archive)

try:
    from huggingface_hub import HfApi
    token = get_secret("HF_TOKEN")
    api = HfApi(token=token)
    repo = f"{api.whoami()['name']}/safevision-artifacts"
    api.create_repo(repo, private=True, exist_ok=True)
    api.upload_folder(folder_path=str(OUT), repo_id=repo, path_in_repo="nb1x_ppe8_detector")
    print("Uploaded to private HF repo:", repo)
except Exception as e:
    print("Hugging Face upload skipped:", e)

## References

Numbering matches *SafeVision_Paper_Summaries.pdf*. Papers used directly in this notebook are **in bold**.

1. Chen, X., & Zou, Z. (2026). Are large pre-trained vision language models effective construction safety inspectors? *Data-Centric Engineering*. https://doi.org/10.1017/dce.2026.10044 (arXiv:2508.11011)
2. **Ahmad, H. M., & Rahimi, A. (2025). SH17: A dataset for human safety and personal protective equipment detection in manufacturing industry. *Journal of Safety Science and Resilience*, 6(2), 175-185. https://doi.org/10.1016/j.jnlssr.2024.09.002** (dataset, class list, 640 px benchmark, small-object and cross-domain findings)
3. **Wang, Z., Wu, Y., Yang, L., Thirunavukarasu, A., Evison, C., & Zhao, Y. (2021). Fast personal protective equipment detection for real construction sites using deep learning approaches. *Sensors*, 21(10), 3478. https://doi.org/10.3390/s21103478** (CHV dataset, 86.55% in-domain mAP)
4. López, L., Suárez-Ramírez, J., Alemán-Flores, M., & Monzón, N. (2025). Automated PPE compliance monitoring in industrial environments using deep learning-based detection and pose estimation. *Automation in Construction*, 176, 106231. https://doi.org/10.1016/j.autcon.2025.106231 (used in Notebook 2)
12. **Adil, M., Ahmed, M., Aqib, M., Gonzalez, V. A., Lee, G., & Mei, Q. (2026). Integration of object detection and small VLMs for construction safety hazard identification. arXiv:2604.05210.** (YOLO11 training settings: 640 px, 50 epochs, patience 15)
- Jocher, G., Qiu, J., & Chaurasia, A. (2023-2026). *Ultralytics YOLO* (Version 8.x / YOLO11) [Software]. https://github.com/ultralytics/ultralytics · Docs: https://docs.ultralytics.com